## Instalación de Librerías

In [1]:
pip install sentinelhub geopandas pandas numpy matplotlib seaborn jupyter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.4/240.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 65.2 MB/s eta 0:00:00


## Importación de Librerías

In [4]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point
from sentinelhub import (
    SHConfig, BBox, CRS, DataCollection,
    SentinelHubRequest, MimeType, bbox_to_dimensions
)
from dotenv import load_dotenv
import xml.etree.ElementTree as ET

## Configuración de Entorno

In [16]:
load_dotenv()

config = SHConfig()
config.sh_client_id = "786a6cb9-7188-43d2-be17-275c26bf8c23"
config.sh_client_secret = "FkfZcDwwZzYefE6QdKzys0g7JRDH9DpW"

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Lectura del archivo kml

### Funciones auxiliares

In [5]:
def parse_kml(path):
    tree = ET.parse(path)
    root = tree.getroot()
    ns = {'kml': 'http://www.opengis.net/kml/2.2'}

    records = []
    for pm in root.findall('.//kml:Placemark', ns):
        name  = pm.find('kml:name', ns).text
        coords_text = pm.find('.//kml:coordinates', ns).text.strip()
        lon, lat, _ = map(float, coords_text.split(','))
        records.append({'parcela': name, 'geometry': Point(lon, lat)})

    return gpd.GeoDataFrame(records, crs='EPSG:4326')

In [9]:
gdf = parse_kml('/content/drive/MyDrive/MNA/Avocados/jalisco.kml')
print(f"Parcelas cargadas: {len(gdf)}")
gdf.head()

Parcelas cargadas: 100


,parcela,geometry
0,H1,POINT (-103.48747 19.66316)
1,H2,POINT (-103.49168 19.65958)
2,H3,POINT (-103.49595 19.64923)
3,H4,POINT (-103.49342 19.64893)
4,H5,POINT (-103.49054 19.64769)


In [17]:
# --- Celda 3: Crear buffer y extraer NDVI para una parcela (prueba) ---
def get_ndvi_for_point(lon, lat, config,
                        start_date="2024-01-01",
                        end_date="2024-12-31",
                        buffer_deg=0.001):   # ~100m

    bbox = BBox(
        bbox=[lon - buffer_deg, lat - buffer_deg,
              lon + buffer_deg, lat + buffer_deg],
        crs=CRS.WGS84
    )

    evalscript = """
    //VERSION=3
    function setup() {
        return { input: ["B04", "B08", "dataMask"],
                 output: { bands: 1 } };
    }
    function evaluatePixel(s) {
        let ndvi = (s.B08 - s.B04) / (s.B08 + s.B04);
        return [s.dataMask ? ndvi : -999];
    }
    """

    request = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L2A,
            time_interval=(start_date, end_date),
            mosaicking_order='leastCC'   # least cloud cover
        )],
        responses=[SentinelHubRequest.output_response('default', MimeType.TIFF)],
        bbox=bbox,
        size=bbox_to_dimensions(bbox, resolution=10),
        config=config
    )

    return request.get_data()[0]

# Prueba con H1
row = gdf.iloc[0]
ndvi_array = get_ndvi_for_point(
    row.geometry.x, row.geometry.y, config
)
print(f"Shape: {ndvi_array.shape}")
print(f"NDVI promedio H1: {ndvi_array[ndvi_array != -999].mean():.3f}")

Shape: (22, 21)
NDVI promedio H1: 95.985
